# Bias-Aware Self-Learning — Versión Corregida

**Cambios respecto a la versión anterior:**

1. **Test set = población oracle** (aceptados + rechazados con labels reales), no solo aceptados del holdout.
2. **Patience=2** en el early stopping: para de iterar solo si no mejora 2 veces seguidas.
3. **Umbrales absolutos** para el labeling de rechazados: `threshold_low=0.3` y `threshold_high=0.6` en lugar de percentiles sobre el pool (que siempre daban prob=1.0 por la composición sesgada de los rechazados).

In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.calibration import CalibratedClassifierCV
from scipy.stats import loguniform
import xgboost as xgb
from sklearn.model_selection import ParameterGrid

# PREPARACIÓN DE DATOS

In [26]:
def load_data(path):
    df = pd.read_csv(path)
    if "Unnamed: 0" in df.columns[0]:
        df = df.drop(columns=df.columns[0])
    return df

def scale_numericals(X_fit, X_transform_list, num_cols, method="standard"):
    if not num_cols:
        return [df.copy() for df in X_transform_list]
    scaler = StandardScaler() if method == "standard" else MinMaxScaler()
    scaler.fit(X_fit[num_cols])
    transformed_dfs = []
    for X in X_transform_list:
        scaled_data = scaler.transform(X[num_cols])
        df_scaled = pd.DataFrame(scaled_data, columns=num_cols, index=X.index)
        transformed_dfs.append(df_scaled)
    return transformed_dfs

In [27]:
def prepare_data_bias_aware_synthetic(config, path_labeled, path_unlabeled, path_oracle=None):
    """
    FIX 1: Ahora también carga el oracle dataset para usar como test set poblacional.
    """
    df_labeled = load_data(path_labeled)
    X_unlabeled = load_data(path_unlabeled)

    target_col = config.get("target", "y")
    split_col = "split"

    y_labeled = df_labeled[target_col].copy()
    split_labeled = df_labeled[split_col].copy()
    X_labeled = df_labeled.drop(columns=[target_col, split_col])
    X_unlabeled = X_unlabeled[X_labeled.columns]
    num_cols = X_labeled.columns.tolist()
    train_idx = split_labeled == "train"
    X_fit_num = X_labeled.loc[train_idx, num_cols]

    transform_list = [X_labeled, X_unlabeled]
    df_oracle = None
    y_oracle = None

    if path_oracle is not None:
        df_oracle_raw = load_data(path_oracle)
        y_oracle = df_oracle_raw[target_col].copy()
        X_oracle = df_oracle_raw[num_cols]
        transform_list.append(X_oracle)

    processed = scale_numericals(
        X_fit_num, transform_list, num_cols,
        method=config.get("numeric_transform", "standard")
    )

    X_labeled_final = processed[0]
    X_unlabeled_final = processed[1]

    df_labeled_processed = X_labeled_final.copy()
    df_labeled_processed["split"] = split_labeled.values
    df_labeled_processed[target_col] = y_labeled.values

    result = {
        "df_labeled": df_labeled_processed,
        "X_unlabeled": X_unlabeled_final,
        "feature_names": X_labeled_final.columns.tolist(),
        "target_col": target_col,
        "split_col": split_col
    }

    if path_oracle is not None:
        result["X_oracle"] = processed[2]
        result["y_oracle"] = y_oracle.values

    return result

# Ejecución de la preparación de datos

In [28]:
config = {
    "target": "y",
    "numeric_transform": "standard",
    "categorical_encoding": "ohe"
}

path_acc     = "data/accepted_labeled.csv"
path_rej     = "data/rejected_unlabeled.csv"
path_oracle  = "data/oracle_population.csv"   # FIX 1: dataset poblacional

data_prepared = prepare_data_bias_aware_synthetic(
    config,
    path_acc,
    path_rej,
    path_oracle=path_oracle
)

# Limpiar NaNs
data_prepared["df_labeled"] = data_prepared["df_labeled"].dropna()
data_prepared["X_unlabeled"] = data_prepared["X_unlabeled"].dropna()

print("df_labeled shape:", data_prepared["df_labeled"].shape)
print("X_unlabeled shape:", data_prepared["X_unlabeled"].shape)
if "X_oracle" in data_prepared:
    print("X_oracle shape:", data_prepared["X_oracle"].shape)
    print(f"Bad rate oracle: {data_prepared['y_oracle'].mean():.2%}")

df_labeled shape: (14111, 7)
X_unlabeled shape: (5889, 5)
X_oracle shape: (20000, 5)
Bad rate oracle: 44.82%


# Bias-Aware Self-Learning

In [29]:
def bayesian_evaluation(model, X_accepts, y_accepts, X_rejects, prior_prob_model, n_iterations=100):
    """
    Bayesian Evaluation Framework (Algoritmo 1).
    Simula etiquetas para rechazados y calcula AUC esperado sobre población completa.
    """
    pred_accepts = model.predict_proba(X_accepts)[:, 1]
    pred_rejects = model.predict_proba(X_rejects)[:, 1]
    rejects_priors = prior_prob_model.predict_proba(X_rejects)[:, 1]

    scores = []
    for j in range(n_iterations):
        y_rejects_simulated = np.random.binomial(1, rejects_priors)
        y_combined = np.concatenate([y_accepts, y_rejects_simulated])
        pred_combined = np.concatenate([pred_accepts, pred_rejects])
        if len(np.unique(y_combined)) < 2:
            continue
        scores.append(roc_auc_score(y_combined, pred_combined))

    return np.mean(scores) if scores else 0.5

In [30]:
def filtering_step(X_accepts, X_rejects, beta_l=0.05, beta_u=0.95):
    """
    Filtra rechazados por similaridad con aceptados (propensity score).
    Elimina los muy distintos (posible outlier) y los muy similares (redundantes).
    """
    X_sim = pd.concat([X_accepts, X_rejects])
    y_sim = np.concatenate([np.ones(len(X_accepts)), np.zeros(len(X_rejects))])

    sim_model = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
    sim_model.fit(X_sim, y_sim)

    prob_similarity = sim_model.predict_proba(X_rejects)[:, 1]
    lower_cut = np.percentile(prob_similarity, beta_l * 100)
    upper_cut = np.percentile(prob_similarity, beta_u * 100)

    mask = (prob_similarity >= lower_cut) & (prob_similarity <= upper_cut)
    X_rejects_filtered = X_rejects[mask].copy()

    print(f"Filtering: {len(X_rejects)} originales -> {len(X_rejects_filtered)} seleccionados para Self-Learning.")
    print(f"Se eliminaron los muy distintos (prob < {lower_cut:.3f}) y los muy similares (prob > {upper_cut:.3f})")

    return X_rejects_filtered

In [31]:
class BiasAwareSelfLearning:
    def __init__(
        self,
        strong_learner_type="logit",
        random_state=42,
        n_iter_search=3,
        cv_folds=3
    ):
        self.random_state = random_state
        self.n_iter_search = n_iter_search
        self.cv_folds = cv_folds
        self.strong_learner_type = strong_learner_type

        self.weak_learner = LogisticRegression(
            l1_ratio=0.5,
            penalty="elasticnet",
            solver="saga",
            random_state=random_state,
            max_iter=5000
        )
        self.calibrated_weak_learner = None
        self.best_model = None

        # En __init__:
        self.X_augmented = None
        self.y_augmented = None

        self.best_basl_instance = None


        self.xgb_param_dist = {
            "n_estimators": [50, 100, 200],
            "max_depth": [3, 4, 5],
            "learning_rate": [0.01, 0.05, 0.1],
            "subsample": [0.8, 0.9, 1.0],
            "colsample_bytree": [0.8, 0.9, 1.0],
            "gamma": [0, 0.1, 0.5],
            "reg_alpha": [0, 0.1, 1]
        }

    def _fit_and_calibrate_once(self, X_train, y_train, X_calib, y_calib):
        X_combined = pd.concat([X_train, X_calib], axis=0, ignore_index=True)
        y_combined = pd.concat([y_train, y_calib], axis=0, ignore_index=True)
        n_train = len(X_train)
        custom_cv = [(np.arange(0, n_train), np.arange(n_train, len(X_combined)))]
        calibrated_model = CalibratedClassifierCV(
            estimator=self.weak_learner,
            method='isotonic',
            cv=custom_cv
        )
        calibrated_model.fit(X_combined, y_combined)
        return calibrated_model

    def _optimize_strong_learner(self, X, y):
        y = pd.Series(y).astype(int).values

        if self.strong_learner_type == "xgb":
            base_model = xgb.XGBClassifier(
                eval_metric="logloss",
                random_state=self.random_state
            )
            param_dist = self.xgb_param_dist

        elif self.strong_learner_type == "logit":
            base_model = LogisticRegression(
                max_iter=1000,
                random_state=self.random_state
            )
            param_dist = {
                "C": [0.01, 0.1, 1, 10],
                "penalty": ["l2"],
                "solver": ["lbfgs"]
            }

        elif self.strong_learner_type == "rf":
            base_model = RandomForestClassifier(random_state=self.random_state)
            param_dist = {
                "n_estimators": [100, 200, 300],
                "max_depth": [3, 5, 8, None],
                "min_samples_leaf": [1, 5, 10],
                "max_features": ["sqrt", "log2"],
                "scale_pos_weight": [2, 3, 4]
            }
        else:
            raise ValueError("strong_learner_type debe ser: 'xgb', 'logit' o 'rf'")

        search = RandomizedSearchCV(
            estimator=base_model,
            param_distributions=param_dist,
            n_iter=self.n_iter_search,
            scoring="roc_auc",
            cv=self.cv_folds,
            random_state=self.random_state,
            n_jobs=-1,
            verbose=0
        )
        search.fit(X, y)
        print(f"    [HPO] Strong learner: {self.strong_learner_type}")
        print(f"    [HPO] Mejores params: {search.best_params_}")
        print(f"    [HPO] AUC en CV: {search.best_score_:.4f}")
        return search.best_estimator_

    def fit(
        self, df_labeled, X_unlabeled, target_col, split_col,
        y_unlabeled_ground_truth=None,
        beta_l=0.05, beta_u=0.95,
        rho=0.1, gamma=0.05, theta=0.95,
        max_iter=20,
        patience=3,
        pct_low=20,
        pct_high=60,
        X_holdout=None,
        y_holdout=None
    ):
        train_mask = df_labeled[split_col] == 'train'
        calib_mask = df_labeled[split_col] == 'calibration'
        test_mask  = df_labeled[split_col] == 'test'

        X_train = df_labeled[train_mask].drop([target_col, split_col], axis=1)
        y_train = df_labeled[train_mask][target_col]
        X_calib = df_labeled[calib_mask].drop([target_col, split_col], axis=1)
        y_calib = df_labeled[calib_mask][target_col]
        X_test  = df_labeled[test_mask].drop([target_col, split_col], axis=1)
        y_test  = df_labeled[test_mask][target_col]

        print("--- Etapa 1: Filtering ---")
        pool_rejects = filtering_step(X_train, X_unlabeled, beta_l, beta_u)

        print("\n--- Etapa 2: Entrenando Weak Learner (Estático) ---")
        self.calibrated_weak_learner = self._fit_and_calibrate_once(
            X_train, y_train, X_calib, y_calib
        )

        print("\n--- Etapa 3: Optimizando Strong Learner Base ---")
        self.strong_learner = self._optimize_strong_learner(X_train, y_train)
        self.best_model = self.strong_learner

        if X_holdout is not None and y_holdout is not None:
            preds_h = self.strong_learner.predict_proba(X_holdout)[:, 1]
            best_metric = roc_auc_score(y_holdout, preds_h)
        else:
            best_metric = bayesian_evaluation(
                self.strong_learner, X_test, y_test,
                pool_rejects, self.calibrated_weak_learner
            )
        print(f"Métrica Bayesiana Inicial: {best_metric:.4f}")

        X_augmented = X_train.copy()
        y_augmented = y_train.copy()

        print("\n--- Etapa 4: Ciclo Self-Learning ---")
        no_improve_count = 0  # FIX 2: contador para patience

        for i in range(max_iter):
            print(f"\nIteración {i+1}:")

            if len(pool_rejects) == 0:
                print("  Pool vacío. Fin.")
                break

            # Muestreo aleatorio del pool
            sample_size = int(len(pool_rejects) * rho)
            if sample_size < 10:
                sample_size = len(pool_rejects)
            candidates = pool_rejects.sample(
                n=sample_size, random_state=self.random_state + i
            )

            # Inspección ground truth (opcional, solo para debug)
            if y_unlabeled_ground_truth is not None:
                real_labels = y_unlabeled_ground_truth.loc[candidates.index]
                real_ones = real_labels.sum()
                real_zeros = len(real_labels) - real_ones
                print(f"  [REALIDAD OCULTA] Candidatos ({len(candidates)}): "
                      f"Default={real_ones} ({real_ones/len(candidates):.1%}), "
                      f"Pago={real_zeros} ({real_zeros/len(candidates):.1%})")

            # FIX 3: Labeling con umbrales absolutos
            probs = self.calibrated_weak_learner.predict_proba(candidates)[:, 1]

            cut_low  = np.percentile(probs, pct_low)
            cut_high = np.percentile(probs, pct_high)
            idx_zeros = candidates.index[probs <= cut_low]
            idx_ones  = candidates.index[probs >= cut_high]

            print(f"  [PREDICCIÓN MODELO]")
            print(f"     -> Umbral Bajo (<={cut_low:.3f}): {len(idx_zeros)} etiquetados como 0")
            print(f"     -> Umbral Alto (>={cut_high:.3f}): {len(idx_ones)} etiquetados como 1")

            if len(idx_zeros) == 0 and len(idx_ones) == 0:
                print("  Ningún candidato seleccionado con umbrales actuales.")
                no_improve_count += 1
                if no_improve_count >= patience:
                    print(f"  Early stopping por patience={patience}.")
                    break
                continue

            # Data Augmentation
            X_new = pd.concat([candidates.loc[idx_zeros], candidates.loc[idx_ones]])
            y_new = pd.concat([
                pd.Series(0, index=idx_zeros),
                pd.Series(1, index=idx_ones)
            ])
            X_augmented = pd.concat([X_augmented, X_new])
            y_augmented = pd.concat([y_augmented, y_new])
            pool_rejects = pool_rejects.drop(X_new.index)

            # Re-entrenamiento
            print("  Re-entrenando Strong Learner con datos aumentados...")
            new_strong_learner = self._optimize_strong_learner(X_augmented, y_augmented)

           
            # Evaluación — holdout externo si disponible, sino bayesiana
            if X_holdout is not None and y_holdout is not None:
                preds_h = new_strong_learner.predict_proba(X_holdout)[:, 1]
                current_metric = roc_auc_score(y_holdout, preds_h)
            elif len(pool_rejects) > 0:
                current_metric = bayesian_evaluation(
                    new_strong_learner, X_test, y_test,
                    pool_rejects, self.calibrated_weak_learner
                )
            else:
                preds_test = new_strong_learner.predict_proba(X_test)[:, 1]
                current_metric = roc_auc_score(y_test, preds_test)

            print(f"  Métrica Bayesiana: {current_metric:.4f} (vs mejor: {best_metric:.4f})")

            if current_metric > best_metric:
                best_metric = current_metric
                self.best_model = new_strong_learner
                no_improve_count = 0
                print("  -> Mejora confirmada.")
            else:
                no_improve_count += 1
                print(f"  -> No hubo mejora ({no_improve_count}/{patience}).")
                if no_improve_count >= patience:  # FIX 2
                    print(f"  Early stopping por patience={patience}.")
                    break
        
        # Al final del fit(), antes del return:
        self.X_augmented = X_augmented.copy()
        self.y_augmented = y_augmented.copy()

        return self.best_model

    def fit_with_param_search(
        self, df_labeled, X_unlabeled, target_col, split_col,
        param_grid, y_unlabeled_ground_truth=None, max_iter=100,
        patience=2,
        # FIX 1: test set poblacional (oracle)
        X_test_population=None, y_test_population=None,
        X_holdout=None, y_holdout=None
    ):
        results = []
        best_auc = -np.inf
        best_model = None
        best_params = None

        for params in ParameterGrid(param_grid):
            print("\n" + "="*60)
            print("Probando combinación:")
            print(params)

            model = BiasAwareSelfLearning(
                strong_learner_type=self.strong_learner_type,
                random_state=self.random_state,
                n_iter_search=self.n_iter_search,
                cv_folds=self.cv_folds
            )

            fitted_model = model.fit(
                df_labeled=df_labeled,
                X_unlabeled=X_unlabeled,
                target_col=target_col,
                split_col=split_col,
                y_unlabeled_ground_truth=y_unlabeled_ground_truth,
                beta_l=params["beta_l"],
                beta_u=params["beta_u"],
                rho=params["rho"],
                gamma=params["gamma"],
                theta=params["theta"],
                max_iter=max_iter,
                patience=patience,
                pct_low=params.get("pct_low", 20),
                pct_high=params.get("pct_high", 60),
                X_holdout=X_holdout,
                y_holdout=y_holdout
            )

            # FIX 1: evaluar en población oracle si está disponible
            if X_test_population is not None and y_test_population is not None:
                preds = fitted_model.predict_proba(X_test_population)[:, 1]
                auc = roc_auc_score(y_test_population, preds)
                print(f"AUC test (población oracle): {auc:.4f}")
            else:
                test_mask = df_labeled[split_col] == "test"
                X_test = df_labeled[test_mask].drop([target_col, split_col], axis=1)
                y_test = df_labeled[test_mask][target_col]
                preds = fitted_model.predict_proba(X_test)[:, 1]
                auc = roc_auc_score(y_test, preds)
                print(f"AUC test (solo aceptados): {auc:.4f}")

            results.append({**params, "auc": auc})

            if auc > best_auc:
                best_auc = auc
                best_model = fitted_model
                best_params = params
                self.best_basl_instance = model

        results_df = pd.DataFrame(results).sort_values("auc", ascending=False)
        print("\n" + "="*60)
        print("MEJORES PARÁMETROS")
        print(best_params)
        print(f"MEJOR AUC: {best_auc:.4f}")
        print("="*60)

        self.best_model = best_model
        return best_model

# Ejecución del modelo Bias-Aware

In [32]:
target_col = "y"
split_col  = "split"

param_grid = {
    "beta_l":   [0.01, 0.05],
    "beta_u":   [0.80, 0.85],
    "rho":      [0.20, 0.30],
    "gamma":    [0.05],
    "theta":    [0.90],
    "pct_low":  [15, 25],
    "pct_high": [55, 65],  # captura ~65% como 1
}
bias_aware_model = BiasAwareSelfLearning(
    strong_learner_type="xgb",
    random_state=42,
    n_iter_search=5,             # más iteraciones HPO
    cv_folds=3
)




In [33]:

# FIX 1: pasar oracle como test set poblacional
X_oracle = data_prepared.get("X_oracle", None)
y_oracle = data_prepared.get("y_oracle", None)


print("\n--- Entrenando Modelo Bias-Aware (Self-Learning) ---")
best_bias_model = bias_aware_model.fit_with_param_search(
    df_labeled=data_prepared["df_labeled"],
    X_unlabeled=data_prepared["X_unlabeled"],
    target_col=data_prepared["target_col"],
    split_col=data_prepared["split_col"],
    param_grid=param_grid,
    max_iter=15,
    patience=2,
    X_test_population=X_oracle,
    y_test_population=y_oracle
)


--- Entrenando Modelo Bias-Aware (Self-Learning) ---

Probando combinación:
{'beta_l': 0.01, 'beta_u': 0.8, 'gamma': 0.05, 'pct_high': 55, 'pct_low': 15, 'rho': 0.2, 'theta': 0.9}
--- Etapa 1: Filtering ---
Filtering: 5889 originales -> 4652 seleccionados para Self-Learning.
Se eliminaron los muy distintos (prob < 0.156) y los muy similares (prob > 0.662)

--- Etapa 2: Entrenando Weak Learner (Estático) ---

--- Etapa 3: Optimizando Strong Learner Base ---
    [HPO] Strong learner: xgb
    [HPO] Mejores params: {'subsample': 0.9, 'reg_alpha': 1, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.01, 'gamma': 0.5, 'colsample_bytree': 0.9}
    [HPO] AUC en CV: 0.5354
Métrica Bayesiana Inicial: 0.5508

--- Etapa 4: Ciclo Self-Learning ---

Iteración 1:
  [PREDICCIÓN MODELO]
     -> Umbral Bajo (<=0.326): 168 etiquetados como 0
     -> Umbral Alto (>=0.388): 653 etiquetados como 1
  Re-entrenando Strong Learner con datos aumentados...
    [HPO] Strong learner: xgb
    [HPO] Mejores p

# Comparación final: Baseline vs Bias-Aware

Ambos modelos se evalúan sobre la **población oracle** (aceptados + rechazados con labels reales),
que representa la distribución real P_XY con bad rate ~20%.

In [34]:
df_final   = data_prepared["df_labeled"]
mask_train = df_final[split_col] == 'train'

X_train_base = df_final[mask_train].drop([target_col, split_col], axis=1)
y_train_base = df_final[mask_train][target_col]

# Modelo baseline: logit entrenado solo en aceptados
baseline_model = LogisticRegression(max_iter=5000, random_state=42)
baseline_model.fit(X_train_base, y_train_base)

# FIX 1: evaluar sobre oracle (población completa)
if X_oracle is not None:
    X_eval = X_oracle
    y_eval = y_oracle
    label_eval = "Oracle (población completa)"
else:
    mask_test = df_final[split_col] == 'test'
    X_eval = df_final[mask_test].drop([target_col, split_col], axis=1)
    y_eval = df_final[mask_test][target_col]
    label_eval = "Test (solo aceptados) — instala oracle para comparación correcta"

preds_base = baseline_model.predict_proba(X_eval)[:, 1]
preds_bias = best_bias_model.predict_proba(X_eval)[:, 1]

auc_base = roc_auc_score(y_eval, preds_base)
auc_bias = roc_auc_score(y_eval, preds_bias)

print("\n" + "="*55)
print(f"RESULTADOS — Test set: {label_eval}")
print("="*55)
print(f"1. Baseline (Solo Aceptados):              {auc_base:.5f}")
print(f"2. Bias-Aware (Aceptados + Rechazados):    {auc_bias:.5f}")
print("-"*55)
mejora = (auc_bias - auc_base) * 100
if mejora > 0:
    print(f"ÉXITO: Bias-Aware mejoró en +{mejora:.2f} pp")
else:
    print(f"Sin mejora: {mejora:.2f} pp — revisar hiperparámetros.")
print("="*55)


RESULTADOS — Test set: Oracle (población completa)
1. Baseline (Solo Aceptados):              0.59395
2. Bias-Aware (Aceptados + Rechazados):    0.62526
-------------------------------------------------------
ÉXITO: Bias-Aware mejoró en +3.13 pp


In [35]:
df_tilde = bias_aware_model.best_basl_instance.X_augmented.copy()
df_tilde["y"] = bias_aware_model.best_basl_instance.y_augmented.values
df_tilde.to_csv("data/D_tilde.csv", index=False)
print(f"D_tilde: {len(df_tilde):,} obs, bad rate: {df_tilde['y'].mean():.1%}")

D_tilde: 11,078 obs, bad rate: 43.8%


In [36]:
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
import numpy as np
import pandas as pd

df = data_prepared["df_labeled"]
mask_train = df["split"] == "train"
mask_calib = df["split"] == "calibration"

X_tr = df[mask_train].drop(["y", "split"], axis=1)
y_tr = df[mask_train]["y"]
X_ca = df[mask_calib].drop(["y", "split"], axis=1)
y_ca = df[mask_calib]["y"]

weak = LogisticRegression(l1_ratio=0.5, penalty="elasticnet",
                          solver="saga", random_state=42, max_iter=5000)
X_comb = pd.concat([X_tr, X_ca])
y_comb = pd.concat([y_tr, y_ca])
cv = [(np.arange(len(X_tr)), np.arange(len(X_tr), len(X_comb)))]
cal = CalibratedClassifierCV(estimator=weak, method="isotonic", cv=cv)
cal.fit(X_comb, y_comb)

probs_r = cal.predict_proba(data_prepared["X_unlabeled"])[:, 1]
print(f"Prob media:        {probs_r.mean():.3f}")
print(f"Percentil 5:       {np.percentile(probs_r,  5):.3f}")
print(f"Percentil 10:      {np.percentile(probs_r, 10):.3f}")
print(f"Percentil 25:      {np.percentile(probs_r, 25):.3f}")
print(f"% con prob < 0.35: {(probs_r < 0.35).mean():.1%}")
print(f"% con prob < 0.45: {(probs_r < 0.45).mean():.1%}")
print(f"% con prob < 0.50: {(probs_r < 0.50).mean():.1%}")
print(f"% con prob > 0.52: {(probs_r > 0.52).mean():.1%}")
print(f"% con prob > 0.55: {(probs_r > 0.55).mean():.1%}")

Prob media:        0.391
Percentil 5:       0.318
Percentil 10:      0.318
Percentil 25:      0.326
% con prob < 0.35: 25.6%
% con prob < 0.45: 93.9%
% con prob < 0.50: 94.1%
% con prob > 0.52: 5.0%
% con prob > 0.55: 4.9%
